## SPACEFLOW

In [ ]:
conda create -n sti_spaceflow python=3.10 -y
conda activate sti_spaceflow
pip install SpaceFlow
pip install scanpy anndata pandas numpy scipy matplotlib

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from SpaceFlow import SpaceFlow

BASE = Path.home() / "MERFISH" / "S1R1" / "STI_results"
OUT = BASE / "SpaceFlow"
OUT.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(BASE / "S1R1_prepared.h5ad")

# SpaceFlow should receive the original count matrix here.
adata.X = adata.layers["counts"].copy()

# Build and run the model.
sf = SpaceFlow(adata=adata)
sf.preprocessing_data(n_top_genes=min(3000, adata.n_vars))

sf.train(
    spatial_regularization_strength=0.1,
    z_dim=50,
    lr=1e-3,
    epochs=500,
    max_patience=50,
    min_stop=100,
    random_seed=42,
    gpu=0,  # Use CPU if no compatible GPU is available.
)

# Spatial domains.
sf.segmentation(
    domain_label_save_filepath=str(OUT / "domains.tsv"),
    n_neighbors=20,
    resolution=1.0,
)

sf.plot_segmentation(
    segmentation_figure_save_filepath=str(OUT / "spatial_domains.pdf")
)

# Pseudo-Spatiotemporal Map.
sf.pseudo_Spatiotemporal_Map(
    pSM_values_save_filepath=str(OUT / "pSM_values.tsv"),
    n_neighbors=20,
    resolution=1.0,
)

sf.plot_pSM(
    pSM_figure_save_filepath=str(OUT / "pSM.pdf")
)

print("SpaceFlow completed. Results:", OUT)


## stLearn

In [ ]:
conda create -n sti_stlearn python=3.10 -y
conda activate sti_stlearn
pip install stlearn scanpy anndata pandas numpy scipy matplotlib

In [ ]:

from pathlib import Path
import numpy as np
import scanpy as sc
import stlearn as st
import matplotlib.pyplot as plt

BASE = Path.home() / "MERFISH" / "S1R1" / "STI_results"
OUT = BASE / "stLearn"
OUT.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(BASE / "S1R1_prepared.h5ad")
adata.X = adata.layers["counts"].copy()

# stLearn can perform expression-based trajectory inference
# without using the microscopy images.
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(
    adata, n_top_genes=min(2000, adata.n_vars), flavor="seurat"
)
adata = adata[:, adata.var["highly_variable"]].copy()

sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=min(30, adata.n_vars - 1))
sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca")
sc.tl.leiden(adata, resolution=0.5, key_added="leiden")

print("Available clusters:", adata.obs["leiden"].value_counts().to_dict())

# Plot clusters in actual tissue coordinates to help choose a root.
xy = adata.obsm["spatial"]
fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(
    xy[:, 0], xy[:, 1],
    c=adata.obs["leiden"].astype("category").cat.codes,
    s=3, cmap="tab20"
)
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("stLearn input clusters — select a root cluster")
fig.colorbar(scatter, ax=ax, label="Encoded cluster")
fig.tight_layout()
fig.savefig(OUT / "01_clusters.png", dpi=200)
plt.show()

# IMPORTANT: change this after examining the cluster plot.
ROOT_CLUSTER = "0"

if ROOT_CLUSTER not in adata.obs["leiden"].astype(str).values:
    raise ValueError("ROOT_CLUSTER not present in the Leiden labels.")

# stLearn's set_root selects a root cell from the specified cluster.
adata.uns["iroot"] = st.spatial.trajectory.set_root(
    adata,
    use_label="leiden",
    cluster=ROOT_CLUSTER,
    use_raw=False,
)

# eps is in the coordinate units used by the method's spatial workflow.
# Start with a conservative value and tune to your coordinate scale.
st.spatial.trajectory.pseudotime(
    adata,
    use_label="leiden",
    eps=20,
    n_neighbors=25,
    use_rep="X_pca",
    pseudotime_key="dpt_pseudotime",
)

# Save the cell-level pseudotime.
adata.obs.to_csv(OUT / "cell_results.csv")
adata.write_h5ad(OUT / "stlearn_results.h5ad")

# Plot pseudotime directly on the tissue coordinates.
pt = adata.obs["dpt_pseudotime"].to_numpy()
fig, ax = plt.subplots(figsize=(8, 7))
scat = ax.scatter(xy[:, 0], xy[:, 1], c=pt, s=3, cmap="viridis")
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("stLearn pseudotime — S1R1")
fig.colorbar(scat, ax=ax, label="Pseudotime")
fig.tight_layout()
fig.savefig(OUT / "02_pseudotime.png", dpi=200)
plt.show()

print("stLearn results saved to:", OUT)


## spTrack

In [ ]:
conda create -n sti_spatrack python=3.10 -y
conda activate sti_spatrack
pip install spaTrack scanpy anndata pandas numpy scipy matplotlib

In [ ]:

from pathlib import Path
import numpy as np
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
import spaTrack as spt

BASE = Path.home() / "MERFISH" / "S1R1" / "STI_results"
OUT = BASE / "spaTrack"
OUT.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(BASE / "S1R1_prepared.h5ad")

# spaTrack expects spatial coordinates in X_spatial.
adata.obsm["X_spatial"] = adata.obsm["spatial"].copy()

# Work from original counts, then perform standard preprocessing.
adata.X = adata.layers["counts"].copy()
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Generate provisional clusters to help define a starting state.
sc.pp.highly_variable_genes(
    adata, n_top_genes=min(2000, adata.n_vars), flavor="seurat"
)
adata = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=min(30, adata.n_vars - 1))
sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca")
sc.tl.leiden(adata, resolution=0.5, key_added="cluster")

print(adata.obs["cluster"].value_counts())

# Plot clusters before selecting the root.
xy = adata.obsm["X_spatial"]
fig, ax = plt.subplots(figsize=(8, 7))
scat = ax.scatter(
    xy[:, 0], xy[:, 1],
    c=adata.obs["cluster"].astype("category").cat.codes,
    s=3, cmap="tab20"
)
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("spaTrack: provisional clusters")
fig.colorbar(scat, ax=ax, label="Encoded cluster")
fig.tight_layout()
fig.savefig(OUT / "01_clusters.png", dpi=200)
plt.show()

# Change this after inspecting the plot and cluster composition.
START_CLUSTER = "0"

if START_CLUSTER not in adata.obs["cluster"].astype(str).values:
    raise ValueError("START_CLUSTER not found.")

# The package supports automatic assessment of starting clusters.
adata = spt.assess_start_cluster(adata)
spt.assess_start_cluster_plot(adata)
plt.savefig(OUT / "02_start_cluster_entropy.png", dpi=200,
            bbox_inches="tight")
plt.show()

# Set the starting cells using the selected cluster.
start_cells = spt.set_start_cells(
    adata, select_way="cell_type", cell_type=START_CLUSTER
)

# Optimal-transport-based transition matrix.
adata.obsp["trans"] = spt.get_ot_matrix(
    adata,
    data_type="spatial",
    alpha1=0.5,
    alpha2=0.5,
)

# Pseudotime and spatial velocity field.
adata.obs["ptime"] = spt.get_ptime(adata, start_cells)

adata.uns["E_grid"], adata.uns["V_grid"] = spt.get_velocity(
    adata, basis="spatial", n_neigh_pos=30
)

# Save the outputs.
adata.obs.to_csv(OUT / "cell_results.csv")
adata.write_h5ad(OUT / "spatrack_results.h5ad")

transition = adata.obsp["trans"]
sp.save_npz(OUT / "transition_matrix.npz", transition)

# Plot pseudotime in tissue coordinates.
fig, ax = plt.subplots(figsize=(8, 7))
scat = ax.scatter(
    xy[:, 0], xy[:, 1],
    c=adata.obs["ptime"], s=3, cmap="viridis"
)
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("spaTrack pseudotime")
fig.colorbar(scat, ax=ax, label="Pseudotime")
fig.tight_layout()
fig.savefig(OUT / "03_pseudotime.png", dpi=200)
plt.show()

# Plot the inferred spatial vector field.
fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(xy[:, 0], xy[:, 1], c="lightgray", s=2, alpha=0.5)

E = adata.uns["E_grid"]
V = adata.uns["V_grid"]
ax.quiver(E[0], E[1], V[0], V[1], color="red", angles="xy")
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("spaTrack inferred spatial vector field")
fig.tight_layout()
fig.savefig(OUT / "04_vector_field.png", dpi=200)
plt.show()

print("spaTrack results saved to:", OUT)


## staVia

In [ ]:
conda create -n sti_stavia python=3.10 -y
conda activate sti_stavia
pip install pyVIA scanpy anndata pandas numpy scipy matplotlib

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import pyVIA.core as via

BASE = Path.home() / "MERFISH" / "S1R1" / "STI_results"
OUT = BASE / "StaVia"
OUT.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(BASE / "S1R1_prepared.h5ad")
adata.X = adata.layers["counts"].copy()

sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(
    adata, n_top_genes=min(2000, adata.n_vars), flavor="seurat"
)
adata = adata[:, adata.var["highly_variable"]].copy()

# StaVia uses a lower-dimensional expression representation.
n_pcs = min(30, adata.n_vars - 1, adata.n_obs - 1)
sc.tl.pca(adata, n_comps=n_pcs)

# Spatially smooth expression using nearby cells.
coords = adata.obsm["spatial"]
spatial_expression = via.spatial_input(
    X_genes=adata.X,
    spatial_coords=coords,
    knn_spatial=6,
    spatial_weight=0.3,
)

# Use the spatially adjusted expression for the trajectory analysis.
from sklearn.decomposition import PCA
input_embedding = PCA(
    n_components=min(30, spatial_expression.shape[1],
                     spatial_expression.shape[0] - 1),
    random_state=42,
).fit_transform(np.asarray(spatial_expression))

# Initial clusters can help with visualization and root selection.
sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca")
sc.tl.leiden(adata, resolution=0.5, key_added="cluster")
print(adata.obs["cluster"].value_counts())

fig, ax = plt.subplots(figsize=(8, 7))
scat = ax.scatter(
    coords[:, 0], coords[:, 1],
    c=adata.obs["cluster"].astype("category").cat.codes,
    s=3, cmap="tab20"
)
ax.invert_yaxis()
ax.set_aspect("equal")
ax.set_title("StaVia: provisional clusters")
fig.colorbar(scat, ax=ax, label="Encoded cluster")
fig.tight_layout()
fig.savefig(OUT / "01_clusters.png", dpi=200)
plt.show()

# IMPORTANT: choose a biologically plausible root cluster.
# This is a placeholder; it must be reviewed before interpretation.
ROOT_CLUSTER = "0"

if ROOT_CLUSTER not in adata.obs["cluster"].astype(str).values:
    raise ValueError("ROOT_CLUSTER is not present.")

# The cluster labels are provisional, not validated cell types.
labels = adata.obs["cluster"].astype(str).tolist()

v = via.VIA(
    data=input_embedding,
    true_label=labels,
    edgepruning_clustering_resolution=0.3,
    edgepruning_clustering_resolution_local=1,
    knn=15,
    cluster_graph_pruning=0.5,
    neighboring_terminal_states_threshold=4,
    too_big_factor=0.3,
    resolution_parameter=1,
    root_user=[ROOT_CLUSTER],
    dataset="S1R1",
    random_seed=42,
    is_coarse=True,
    preserve_disconnected=True,
    pseudotime_threshold_TS=40,
    x_lazy=0.99,
    alpha_teleport=0.99,
    edgebundle_pruning=0.5,
    edgebundle_pruning_twice=True,
    do_spatial_knn=True,
    do_spatial_layout=True,
    spatial_coords=coords,
    spatial_knn=6,
)

v.run_VIA()

# Save the fitted model and its computed results.
import pickle
with open(OUT / "stavia_model.pkl", "wb") as f:
    pickle.dump(v, f)

# Official graph visualization.
fig, ax1, ax2 = via.plot_piechart_viagraph(
    via_object=v,
    cmap_piechart="rainbow",
    cmap="plasma",
    highlight_terminal_clusters=True,
    show_legend=True,
)
fig.savefig(OUT / "02_trajectory_graph.png", dpi=200,
            bbox_inches="tight")
plt.show()

# Print likely result attributes for version-specific extraction.
print("StaVia object attributes containing 'pt', 'lineage', or 'terminal':")
for name in dir(v):
    if any(term in name.lower()
           for term in ["pt", "lineage", "terminal", "clustergraph"]):
        if not name.startswith("_"):
            print(name)

print("StaVia model saved to:", OUT)


## CASCAT

In [ ]:
conda create -n sti_cascat python=3.10 -y
conda activate sti_cascat

git clone https://github.com/deepomicslab/CASCAT.git
cd CASCAT

# Install the official dependencies for your platform.
# The following commands are from the repository's CUDA 11.8 setup.
pip install torch==2.1.1 torchvision==0.16.1 torchaudio==2.1.1 \
  --index-url https://download.pytorch.org/whl/cu118

pip install torch_geometric==2.6.1 pyg_lib torch_scatter \
  torch_sparse torch_cluster torch_spline_conv \
  -f https://data.pyg.org/whl/torch-2.1.0+cu118.html

pip install scanpy==1.10.1 matplotlib networkx scikit-misc \
  pydot pot numpy==1.26.4 scikit-learn==1.5.2

pip install numba==0.60.0 numba-scipy==0.4.0 \
  pandas==2.2.3 scipy==1.11.0 pyyaml==6.0.3

In [ ]:
from pathlib import Path
import scanpy as sc

base = Path.home() / "MERFISH" / "S1R1" / "STI_results"

adata = sc.read_h5ad(base / "S1R1_prepared.h5ad")

# Use original counts as CASCAT's expression input.
adata.X = adata.layers["counts"].copy()

# Keep spatial coordinates in the AnnData object.
assert "spatial" in adata.obsm
adata.write_h5ad(base / "S1R1_CASCAT_input.h5ad")

print(adata)
print("Saved:", base / "S1R1_CASCAT_input.h5ad")

In [ ]:

from pathlib import Path
import scanpy as sc
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp

RESULT_FILE = Path("./result/data_processed.h5ad")
OUT = Path("./S1R1_CASCAT_plots")
OUT.mkdir(parents=True, exist_ok=True)

adata = sc.read_h5ad(RESULT_FILE)
print(adata)
print("obs columns:", adata.obs.columns.tolist())
print("obsm keys:", list(adata.obsm.keys()))
print("uns keys:", list(adata.uns.keys()))

# Plot predicted clusters in spatial coordinates.
if "spatial" in adata.obsm and "cascat_clusters" in adata.obs:
    xy = adata.obsm["spatial"]
    fig, ax = plt.subplots(figsize=(8, 7))
    scat = ax.scatter(
        xy[:, 0], xy[:, 1],
        c=adata.obs["cascat_clusters"].astype("category").cat.codes,
        s=3, cmap="tab20"
    )
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_title("CASCAT predicted clusters")
    fig.colorbar(scat, ax=ax, label="Encoded cluster")
    fig.tight_layout()
    fig.savefig(OUT / "01_clusters.png", dpi=200)
    plt.show()

# Inspect trajectory connectivity.
conn = adata.uns.get("cascat_connectivities")
if conn is not None:
    if sp.issparse(conn):
        conn = conn.toarray()
    plt.figure(figsize=(8, 7))
    plt.imshow(conn, cmap="magma", aspect="auto")
    plt.colorbar(label="Connectivity weight")
    plt.title("CASCAT trajectory connectivity")
    plt.xlabel("Cluster or graph node")
    plt.ylabel("Cluster or graph node")
    plt.tight_layout()
    plt.savefig(OUT / "02_connectivity.png", dpi=200)
    plt.show()

# Save all cell-level annotations for downstream plotting.
adata.obs.to_csv(OUT / "cell_results.csv")
print("CASCAT plots saved to:", OUT)
